# Preprocessing

In [4]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import config
from llm_setup import llm_model, llm_response

In [5]:
import urllib.request

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.messages import HumanMessage, AIMessage


## Load The Document

In [6]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Download the file using urllib with error handling
try:
    # Check if file exists and remove it
    if os.path.exists(filename):
        os.remove(filename)
        print(f"Removed existing file: {filename}")
    
    print(f"Downloading file from: {url}")
    urllib.request.urlretrieve(url, filename)
    print('File downloaded successfully')
    
    # Verify the file was downloaded
    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        print(f"File size: {os.path.getsize(filename)} bytes")
        
        with open(filename, 'r', encoding='utf-8') as file:
            contents = file.read()
            print("\nFirst 1000 characters of the document:")
            print("=" * 50)
            print(contents[:1000])
            print("=" * 50)
    else:
        raise FileNotFoundError(f"Downloaded file {filename} is empty or doesn't exist")
        
except urllib.error.URLError as e:
    print(f"URL Error: {e}")
    print("Please check your internet connection and try again.")
    raise
except Exception as e:
    print(f"Error downloading file: {e}")
    print("Using a local fallback or exiting...")
    raise

Removed existing file: companyPolicies.txt
File downloaded successfully
File size: 15660 bytes

First 1000 characters of the document:
1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. W

## Splitting the document into chunks

In [7]:
with open(filename) as f:
    text = f.read()
documents = [Document(page_content=text)]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:3]):  # Show first 3 chunks as example
    print(f"\nChunk {i+1} (length: {len(chunk.page_content)}):")
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)


Created 27 chunks

Chunk 1 (length: 18):
1.	Code of Conduct

Chunk 2 (length: 773):
Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respe...

Chunk 3 (length: 850):
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violatio...


## Embedding and storing


In [8]:
print(f"Creating embeddings for {len(chunks)} chunks...")
texts = [chunk.page_content for chunk in chunks]

# Validate API key before creating embeddings
if not config.OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is not set. Please check your .env file.")

try:
    openai_embeddings = OpenAIEmbeddings(
        model=config.EMBEDDING_MODEL,
        openai_api_key=config.OPENROUTER_API_KEY,
        openai_api_base=config.OPENROUTER_BASE_URL,
        dimensions=config.EMBEDDING_DIMENSIONS
    )
    
    print(f"Creating vector store with collection: company-policies")
    chroma_doc_search = Chroma.from_texts(
        texts,
        openai_embeddings,
        collection_name="company-policies"
    )
    
    # Verify the vector store was created
    store_info = chroma_doc_search.get()
    print(f"\nVector store created successfully!")
    print(f"Number of documents stored: {len(store_info['ids'])}")
    print(f"Collection name: {chroma_doc_search._collection.name}")
    
    # Show a sample document
    if store_info['documents']:
        print("\nSample document (first 300 chars):")
        print("=" * 50)
        print(store_info['documents'][0][:300] + "...")
        print("=" * 50)
    
except Exception as e:
    print(f"Error creating embeddings or vector store: {e}")
    print("Check your API key and network connection.")
    raise

Creating embeddings for 27 chunks...
Creating vector store with collection: company-policies


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"



Vector store created successfully!
Number of documents stored: 27
Collection name: company-policies

Sample document (first 300 chars):
1.	Code of Conduct...


# LLM Model Construction

## Model

In [9]:
model = llm_model(params=None)

INFO:llm_setup:Creating LLM model: openai/gpt-4o-mini
INFO:llm_setup:Configuration: temperature=0.5, max_tokens=512
INFO:llm_setup:LLM model created successfully


## Integrating Langchain

In [10]:
# already using prompt template, exercise was pushing for RetrievalQA, wich is deprecated

retriever = chroma_doc_search.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the given context to answer the question. If you don't know, say you don't know. Context: {context}"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(model, prompt)
qa = create_retrieval_chain(retriever, question_answer_chain)

query = "what is mobile policy?"
result = qa.invoke({"input": query})

print(result["answer"])

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The Mobile Phone Policy is a set of standards and expectations that govern the appropriate and responsible usage of mobile devices within an organization. Its purpose is to ensure that employees use mobile phones in a manner that aligns with company values and legal compliance. The policy includes guidelines on acceptable use, security measures, confidentiality, cost management, compliance with laws and regulations, and procedures for reporting lost or stolen devices. Non-compliance with the policy may result in disciplinary actions, including the potential loss of mobile phone privileges.


# Dive Deeper

## Prompt Template

Already created PromptTemplate previsously, exercise was pushing for RetrievalQA, wich is deprecated

In [11]:
query = "Can I eat in company vehicles?"
result = qa.invoke({"input": query})

print(result["answer"])

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The context does not provide specific information about eating in company vehicles. Therefore, I don't know if eating in company vehicles is allowed or not.


## Make the conversation have memory

As the model does not have memory, it won't relate to it as the car
Sample response:

```Based on the Internet and Email Policy, you cannot:
1. Use company-provided internet and email services for non-job-related tasks during work hours, except for limited personal use during non-work hours that does not interfere with work responsibilities.
2. Share your login credentials or passwords with others.
3. Open email attachments or click on links from unknown sources without exercising caution.
4. Transmit confidential information, trade secrets, or sensitive customer data via email without using encryption.
5. Discuss company matters on public forums or social media without discretion.
6. Engage in harassment, discrimination, or distribute offensive or inappropriate content.
7. Violate relevant laws and regulations regarding internet and email usage, including copyright and data protection.
8. Use company internet and email in ways that may lead to disciplinary measures or termination for policy violations.
It is important to adhere to these guidelines to ensure responsible and secure usage of digital communication tools.
´´´

In [12]:
query = "What I cannot do in it?"
result = qa.invoke({"input": query})

print(result["answer"])

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


You cannot engage in the following activities according to the Internet and Email Policy and the Mobile Phone Policy:

1. **Harassment and Inappropriate Content**: You must not use internet and email services for harassment, discrimination, or distributing offensive or inappropriate content.

2. **Excessive Personal Use**: While limited personal use is allowed, it should not interfere with your work responsibilities.

3. **Sharing Passwords**: You should not share your login credentials or passwords with others.

4. **Ignoring Security Protocols**: You must exercise caution with email attachments and links from unknown sources and report any unusual online activity or potential security breaches promptly.

5. **Transmitting Confidential Information Unsecured**: You should avoid sending sensitive company information via unsecured messaging apps or emails without encryption.

6. **Discussing Company Matters Publicly**: You need to be discreet when discussing company matters in public for

In [13]:
# Rewrite follow-up questions into standalone search queries
# so the retriever can handle references like "it"
contextualize_q_system_prompt = (
    "Given the chat history and the latest user question, "
    "formulate a standalone question that can be understood "
    "without the chat history. Do NOT answer the question; "
    "just rewrite it if needed, otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    model,
    chroma_doc_search.as_retriever(),
    contextualize_q_prompt,
)

# Answer using only retrieved context
qa_system_prompt = (
    "Use the given retrieved context to answer the user's question. "
    "If you don't know, say you don't know.\n\n"
    "Context: {context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(model, qa_prompt)
qa_with_memory = create_retrieval_chain(history_aware_retriever, question_answer_chain)

chat_history = []


In [14]:
query = "what is the mobile policy?"
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history,
})
print(result["answer"])
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"]),
])

query = "Can i eat in company vehicles?"
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history,
})
print(result["answer"])
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"]),
])

query = "What I cannot do in it?"
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history,
})
print(result["answer"])
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"]),
])


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. Its purpose is to ensure that employees use mobile phones in a manner consistent with company values and legal compliance. Key aspects of the policy include:

1. **Acceptable Use**: Mobile devices are primarily for work-related tasks, with limited personal use allowed as long as it does not disrupt work obligations.

2. **Security**: Employees must safeguard their mobile devices and access credentials, be cautious when downloading apps or clicking links from unfamiliar sources, and report any security concerns or suspicious activities.

3. **Confidentiality**: Employees should avoid transmitting sensitive company information via unsecured messaging apps or emails and be discreet when discussing company matters in public.

4. **Cost Management**: Personal phone usage should be kept separate from company accounts, and employees must reimbu

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


The provided context does not specify whether eating in company vehicles is allowed or prohibited. Therefore, I don't know if eating in company vehicles is permitted. It would be best to consult your company's specific policies or ask a supervisor for clarification on this matter.


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


In company vehicles, you cannot smoke, as smoking is not permitted in these vehicles to maintain their condition and cleanliness. Other specific prohibitions are not detailed in the provided context, but generally, it is advisable to avoid any activities that could damage the vehicle or violate company policies. For further clarification, you may want to consult your company's specific vehicle usage policy.


## Putting It All Together

Above, we built a conversational RAG pipeline step by step:
- **Chunking & Embedding**: Split the document into chunks and stored vector embeddings in Chroma.
- **Basic RAG Chain**: Retrieved relevant chunks and answered questions using LCEL.
- **Conversation Memory**: Added a history-aware retriever so follow-up questions work correctly.

Now we wrap everything into an interactive agent loop that maintains its own conversation history.

## Wrap it and make it an agent

In [15]:
def qa_agent():
    qa_agent_chat_history = []
    while True:
        qa_agent_query = input("Question: ")
        if qa_agent_query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break
        qa_agent_result = qa_with_memory.invoke({
            "input": qa_agent_query,
            "chat_history": qa_agent_chat_history,
        })
        print("Answer:", qa_agent_result["answer"])
        qa_agent_chat_history.extend([
            HumanMessage(content=qa_agent_query),
            AIMessage(content=qa_agent_result["answer"]),
        ])


In [16]:
qa_agent()  

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Answer: The Mobile Phone Policy outlines the standards and expectations for the appropriate and responsible use of mobile devices within the organization. Key points of the policy include:

1. **Acceptable Use**: Mobile devices should primarily be used for work-related tasks, with limited personal usage allowed as long as it does not disrupt work obligations.

2. **Security**: Employees must safeguard their mobile devices and access credentials, exercise caution when downloading apps or clicking on unfamiliar links, and report any security concerns or suspicious activities.

3. **Confidentiality**: Employees should avoid transmitting sensitive company information via unsecured messaging apps or emails and be discreet when discussing company matters in public spaces.

4. **Cost Management**: Personal phone usage should be kept separate from company accounts, and employees are responsible for reimbursing the company for any personal charges incurred on company-issued phones.

5. **Compli

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Answer: The context provided does not specify any rules regarding eating in company vehicles. Therefore, I don't know if eating in your car is permitted or not. You may want to check your company's specific policies or guidelines regarding this matter.


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Answer: The context does not provide specific information about what you can do in your car, aside from mentioning that smoking is not permitted in company vehicles. Therefore, I don't know what activities are explicitly allowed or prohibited in your car. You may want to refer to your company's policies or guidelines for further clarification.
Answer: Goodbye!
